# Cellular Automata

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

A refresher on **cellular automata (CA)**: a grid of cells that all update *in lockstep* by the
same tiny local rule, looking only at their neighbors. From a handful of rules emerges
startling complexity — Conway's *Game of Life*, Wolfram's *Rule 30/110*, and the
roguelike *cave generator* that turns random noise into organic caverns in a few passes.
The workhorse for organic, "grown-looking" procedural content.

## 1. What & Why

A **cellular automaton** is defined by four things:

1. A **grid** of cells (1D line, 2D lattice, …).
2. A finite set of **states** each cell can be in (often just *alive*/*dead*, `1`/`0`).
3. A **neighborhood** — which nearby cells a cell can "see" (e.g. the 8 surrounding cells).
4. A **transition rule** — given a cell's state and its neighbors' states, what state it takes
   next.

Every cell applies the *same* rule *simultaneously* each tick, reading the **old** grid and
writing a **new** one. That's the whole model. The magic is **emergence**: purely local,
deterministic rules produce global structure nobody coded explicitly — gliders, fractals,
cave systems.

**The problem it solves.** You want content that looks *grown* rather than *placed* — caves,
moss, erosion, fire/fluid spread, crystal growth, organic blobs. Hand-authoring is tedious and
pure random noise looks like static. A CA bridges them: seed with noise, then let a local
smoothing rule carve coherent shapes out of it. Cheap, parallel, and tunable by a couple of
thresholds.

**When to reach for it.** Cave/dungeon interiors (the classic 4-5 rule), terrain detail, spreading
simulations (fire, water, infection, gas), texture/pattern synthesis, and any "neighbor-driven"
dynamic you can phrase as *"a cell becomes X if enough neighbors are X."*

**When not to.** When you need *guaranteed* connectivity or specific structure (a guaranteed path
from entrance to exit), CA alone won't promise it — you post-process (flood-fill, connect regions)
or use BSP/graph methods ([[bsp-dungeon-generation]]). For smooth gradient terrain reach for noise
functions ([[noise-simplex-worley]], [[perlin-noise]]); for hard tile-adjacency constraints use
Wave Function Collapse ([[wave-function-collapse]]).

## 2. Mental Model

Think of a **stadium crowd doing the wave, but every spectator decides independently and at the
same instant** by one rule: *"stand if most of my neighbors are standing, else sit."* No
conductor, no communication beyond glancing left and right — yet a coherent wave ripples across
the stadium. That's a cellular automaton: global pattern from synchronized local decisions.

The single most important discipline: **double-buffering**. You compute the *entire* next
generation from the *current* one, then swap. If you update cells in place, a cell you already
moved corrupts the neighbor count of the cell next to it — you'd be mixing "now" and "next."

```
   step t            step t+1  (all cells decided from t, simultaneously)
  . # . .           . # . .
  . # # .   --rule-> # # # .
  . . # .           . # . .
       └─ each new cell = f(old self, old 8 neighbors)
```

For procedural caves the loop is just: **fill with random noise → apply the smoothing rule a few
times → the noise organizes into walls and open chambers.** Each pass erodes lonely cells and
fills in nearly-enclosed gaps, so blobs round out into caverns.

## 3. Key Concepts

- **Cell & state** — the atom of the grid and the value it holds. Binary (alive/dead) is most
  common; CA can have many states (e.g. *sand/water/empty* in a falling-sand game).
- **Neighborhood** — the cells a rule may read. **Moore** = the 8 surrounding cells (used for caves
  and Life); **von Neumann** = the 4 orthogonal cells. Radius can be larger.
- **Transition rule** — the function from (self, neighbors) → next state. For 2D life-like rules,
  usually a function of the *count* of live neighbors (e.g. Conway's "B3/S23").
- **Generation / tick / step** — one synchronous update of the whole grid.
- **Synchronous update & double buffering** — all cells update at once from the previous grid; you
  must read old, write new. The #1 correctness rule.
- **Boundary conditions** — what neighbors mean at the edge: **toroidal/wrap** (grid wraps around),
  **fixed** (off-grid treated as dead, or as wall — common for caves so borders stay solid), or
  **reflecting**. Changes behavior dramatically.
- **Elementary CA & Wolfram code** — 1D, two states, 3-cell neighborhood → 2^3 = 8 possible
  neighborhoods, each mapped to 0/1 ⇒ 2^8 = **256 rules**, numbered 0–255. Rule 30 (chaotic/random
  looking), Rule 90 (Sierpiński triangle), Rule 110 (Turing-complete).
- **Life-like rule (B/S notation)** — `B` = neighbor counts that *birth* a dead cell, `S` = counts
  that let a live cell *survive*. Conway = **B3/S23**. The cave rule is essentially **B5678/S45678**
  (a cell is a wall if it or ≥5 neighbors are walls).
- **Emergence** — large-scale order that the local rule never mentions: gliders, oscillators,
  fractals, cave chambers.
- **Wolfram's four classes** — long-run behavior: I uniform, II periodic, III chaotic (Rule 30),
  IV complex/structured (Rule 110, Conway). A useful map of what kind of rule you're holding.

## 4. Setup

Cellular automata need nothing exotic — a 2D array and integer arithmetic. We use **NumPy** so the
neighbor count is a single vectorized convolution-style sum instead of a slow Python loop; it ships
with essentially every scientific Python install.

```bash
%pip install -q numpy
```

`matplotlib` is optional (nicer than ASCII for big grids); the examples render to text so the
notebook executes anywhere, and the plotting cell is gated so it's skipped if matplotlib is
absent.

In [ ]:
# Cellular automata need only NumPy. matplotlib is optional (used for a prettier render below).
#   %pip install -q numpy matplotlib
import numpy as np

rng = np.random.default_rng(7)  # seeded for reproducible grids
print("numpy", np.__version__, "- ready")

## 5. Worked Examples

Three self-contained examples, smallest idea first:

1. **Elementary 1D CA (Wolfram rules)** — the simplest possible CA. One rule number defines
   everything; we render Rule 30 (chaos), Rule 90 (Sierpiński fractal), and Rule 110.
2. **Conway's Game of Life** — the canonical 2D life-like rule (B3/S23) and the double-buffering
   pattern, watching a glider walk across the grid.
3. **Cave generation** — the real procedural-generation payoff: random noise + a smoothing rule
   ⇒ organic caverns, the technique behind countless roguelikes.

### Example 1 — Elementary 1D cellular automata (Wolfram rules)

A 1D row of cells, each `0` or `1`. A cell's next state depends on itself and its two neighbors —
a 3-bit pattern with 8 possibilities. A **rule number** 0–255, read as 8 bits, says what each of
those 8 neighborhoods maps to. Stacking each generation below the last draws the CA's history as a
2D picture. Rule 90 famously draws a **Sierpiński triangle** — a fractal from a one-line rule.

In [ ]:
def elementary_ca(rule, width=63, steps=32, single_seed=True):
    """Run a Wolfram elementary CA; return the list of rows (its space-time history)."""
    row = np.zeros(width, dtype=np.uint8)
    if single_seed:
        row[width // 2] = 1            # start from one live cell in the middle
    else:
        row = rng.integers(0, 2, width, dtype=np.uint8)
    rule_bits = [(rule >> i) & 1 for i in range(8)]  # rule number -> 8 output bits
    history = [row.copy()]
    for _ in range(steps - 1):
        left, right = np.roll(row, 1), np.roll(row, -1)  # wrap-around neighbors
        idx = (left << 2) | (row << 1) | right           # 3-bit neighborhood -> 0..7
        row = np.array([rule_bits[i] for i in idx], dtype=np.uint8)
        history.append(row.copy())
    return history

def render(history):
    return "\n".join("".join("#" if c else " " for c in r) for r in history)

for rule in (30, 90, 110):
    print(f"=== Rule {rule} ===")
    print(render(elementary_ca(rule, width=63, steps=24)))
    print()

Rule 90 carves a clean **Sierpiński triangle** (self-similar fractal); Rule 30 looks random and
is genuinely used as a randomness source; Rule 110 grows localized structures and is
*Turing-complete*. Same machinery, wildly different "classes" of behavior — all from 8 bits.

### Example 2 — Conway's Game of Life (the 2D life-like rule)

The most famous CA. Each cell looks at its **8 Moore neighbors** and applies **B3/S23**: a dead
cell with exactly 3 live neighbors is **born**; a live cell with 2 or 3 survives; everything else
dies. We count neighbors with a vectorized sum over 8 shifted copies of the grid (note the
double-buffer: the new grid is computed entirely from the old one), and watch a **glider** —
a 5-cell pattern that walks diagonally forever.

In [ ]:
def life_step(grid):
    """One Game of Life generation (B3/S23) with toroidal wrap, fully vectorized."""
    # Sum the 8 neighbors by rolling the grid in every direction and adding.
    neighbors = sum(
        np.roll(np.roll(grid, dy, 0), dx, 1)
        for dy in (-1, 0, 1) for dx in (-1, 0, 1)
        if not (dy == 0 and dx == 0)
    )
    born = (grid == 0) & (neighbors == 3)
    survive = (grid == 1) & ((neighbors == 2) | (neighbors == 3))
    return (born | survive).astype(np.uint8)

def show(grid):
    return "\n".join("".join("#" if c else "." for c in row) for row in grid)

grid = np.zeros((8, 12), dtype=np.uint8)
grid[1, 2] = grid[2, 3] = grid[3, 1] = grid[3, 2] = grid[3, 3] = 1  # a glider
for gen in range(4):
    print(f"generation {gen}  (live cells: {int(grid.sum())})")
    print(show(grid))
    print()
    grid = life_step(grid)

The glider's shape is preserved while its position shifts down-and-right — structure that *moves*,
emerging from a rule that never mentions movement. The live-cell count stays 5: it's a stable
spaceship. This same `life_step` is the kernel; only the birth/survive sets change between
life-like rules.

### Example 3 — Procedural cave generation

The procedural-generation classic. Fill a grid with **random walls** (~45% density), then repeatedly
apply a **smoothing rule**: *a cell becomes a wall if it is surrounded by ≥5 wall neighbors,
otherwise it opens up* (with off-grid counted as wall so the border stays solid). After a few
passes the noise organizes into rounded chambers connected by passages — organic caves, no
hand-authoring.

In [ ]:
def count_wall_neighbors(walls):
    """For each cell, count wall cells in its 8-neighborhood; off-grid counts as wall."""
    padded = np.pad(walls, 1, mode="constant", constant_values=1)  # border = wall
    return sum(
        padded[1 + dy:1 + dy + walls.shape[0], 1 + dx:1 + dx + walls.shape[1]]
        for dy in (-1, 0, 1) for dx in (-1, 0, 1)
        if not (dy == 0 and dx == 0)
    )

def smooth(walls):
    n = count_wall_neighbors(walls)
    # Birth/keep a wall if >=5 wall neighbors; otherwise carve it open.
    return (n >= 5).astype(np.uint8)

def render_cave(walls):
    return "\n".join("".join("#" if c else " " for c in row) for row in walls)

h, w = 18, 48
walls = (rng.random((h, w)) < 0.45).astype(np.uint8)  # ~45% initial wall density
print("=== initial random noise ===")
print(render_cave(walls))

for i in range(4):
    walls = smooth(walls)
print("\n=== after 4 smoothing passes ===")
print(render_cave(walls))
print(f"\nopen floor: {int((walls == 0).sum())} cells "
      f"({100 * (walls == 0).mean():.0f}% of the map)")

The static noise resolves into connected open chambers ringed by solid rock. Two knobs control the
look: **initial density** (higher ⇒ more rock, tighter caves) and **pass count** (more ⇒ smoother,
blobbier). Production generators add a post-step — flood-fill to find the largest open region and
discard or tunnel to isolated pockets — because CA smoothing alone doesn't *guarantee* every cave
is reachable.

In [ ]:
# Optional: render the cave as an image if matplotlib is available (skipped otherwise).
import importlib.util

if importlib.util.find_spec("matplotlib") is not None:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 2.4))
    ax.imshow(walls, cmap="binary", interpolation="nearest")
    ax.set_title("Cellular-automata cave (wall = black)")
    ax.set_xticks([]); ax.set_yticks([])
    plt.show()
else:
    print("matplotlib not installed - ASCII render above is the output. "
          "Install with: %pip install -q matplotlib")

## 6. Gotchas & Pitfalls

- **Updating in place instead of double-buffering.** If you overwrite cells while still reading
  the same grid, a just-updated cell pollutes its neighbors' counts and you get garbage. Always
  compute the next generation into a *new* array from the *old* one (as every example here does).
- **Boundary handling is a real design choice, not a detail.** Wrap (toroidal) makes gliders fly
  off one edge and reappear on the other; fixed-as-dead lets patterns die at the border;
  **fixed-as-wall** is what you usually want for caves so the map has a solid frame. Pick
  deliberately — it changes the output completely.
- **Wrong initial density or pass count.** Too few wall cells (<40%) ⇒ caves never close up; too
  many (>55%) ⇒ the map fills solid. Too many smoothing passes ⇒ featureless blobs; too few ⇒ still
  noisy. These two numbers are the whole tuning surface — sweep them.
- **No connectivity guarantee.** CA smoothing happily produces isolated pockets the player can't
  reach. If reachability matters, flood-fill after generating and connect or cull stranded regions
  — the CA gives you *shape*, not *topology*.
- **Off-by-one in the rule.** "≥5 neighbors" vs ">5", counting self or not, Moore vs von Neumann —
  small changes flip the aesthetic entirely (and B5/S5 ≠ B5678/S45678). Write the rule explicitly
  and test it on a tiny hand-checkable grid.
- **Slow Python loops.** A nested `for` over every cell is fine for 50×50 but crawls at scale.
  Vectorize the neighbor count with shifts/convolution (`np.roll`, `scipy.signal.convolve2d`, or a
  3×3 kernel) as shown — easily 100× faster.
- **Forgetting reproducibility.** Generation seeds from random noise; without a seeded RNG you
  can't reproduce a map for tests or save-files. Seed it (`np.random.default_rng(seed)`).
- **Expecting precise control.** CA is *emergent* — you nudge it with rules and seeds, you don't
  dictate exact layouts. If you need a specific structure, generate the skeleton another way and
  use CA only for organic detailing.

## 7. When to Use vs Alternatives

**Reach for cellular automata when** you want *organic, grown* content and can express the look as a
local neighbor rule: caves, moss/vegetation spread, erosion, fire/fluid/gas simulation, falling
sand, texture synthesis. You get cheap, parallelizable, highly tunable results from a handful of
lines and two knobs (density + passes), with no training data.

| Approach | Strength | Weakness vs. CA | Use when |
|---|---|---|---|
| **Cellular automata** | Organic emergent shapes; trivial & fast (vectorizable); tiny tunable rule | No structural guarantees (connectivity, paths); emergent ⇒ hard to control precisely | Caves, spreading sims, organic blobs/detail |
| **Noise functions** ([[noise-simplex-worley]], [[perlin-noise]]) | Smooth, continuous, infinite, seekable terrain | No discrete "rooms"/structure; not rule-driven dynamics | Heightmaps, gradients, clouds, biome blends |
| **BSP / room-graph** ([[bsp-dungeon-generation]]) | Guaranteed connected rooms + corridors; controllable layout | Looks rectangular/man-made, not organic | Structured dungeons, buildings, levels |
| **Drunkard's walk** ([[drunkards-walk]]) | Dead-simple connected organic tunnels | Less control over chamber shape/density | Quick winding caves with guaranteed connectivity |
| **Wave Function Collapse** ([[wave-function-collapse]]) | Hard local tile-adjacency constraints | Heavier; can backtrack; needs a tileset | Tile maps with strict compatibility rules |
| **Grammars / L-systems** ([[context-free-grammars]], [[l-systems]]) | Explicit hierarchical/recursive structure | Not for diffuse organic fields | Plants, structured/recursive content |

Rule of thumb: **CA for things that *grow*, BSP for things that are *built*, noise for things that
*flow*.** They compose beautifully — generate caves with CA, *guarantee* connectivity with a
flood-fill + drunkard's-walk tunnel, and scatter decoration with noise.

## 8. Resources

- **"Cellular automaton" — Wikipedia** — definitions, neighborhoods, boundary conditions, and the
  Wolfram classes: <https://en.wikipedia.org/wiki/Cellular_automaton>
- **"Conway's Game of Life" — Wikipedia / LifeWiki** — the canonical rule, patterns (gliders,
  guns), and B/S notation: <https://en.wikipedia.org/wiki/Conway%27s_Game_of_Life> and the pattern
  encyclopedia at <https://conwaylife.com/wiki/>
- **"Generate Random Cave Levels Using Cellular Automata" — RogueBasin** — the classic step-by-step
  cave-generation tutorial: <https://www.roguebasin.com/index.php/Cellular_Automata_Method_for_Generating_Random_Cave-Like_Levels>
- **"Procedural Cave Generation" — Sebastian Lague (video series)** — excellent visual walk-through
  including the flood-fill connectivity step: <https://www.youtube.com/watch?v=v7yyZZjF1z4>
- **Wolfram, *A New Kind of Science* (free online)** — elementary CA, rule numbering, and the four
  classes in exhaustive depth: <https://www.wolframscience.com/nks/>
- **Red Blob Games** — high-signal interactive articles on map generation that pair well with CA:
  <https://www.redblobgames.com/>

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def step(grid, birth, survive, outside="wall"):
    """One synchronous generation: read the old grid, write a new one."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE